# SFT 概念与 Wordle 任务

欢迎来到 Qwen3-1.7B 训练实战教程的第一阶段——监督微调（Supervised Fine-Tuning，SFT）。

本教程的目标是：**训练一个会玩 Wordle 猜词游戏的语言模型**。这是一个两阶段的任务——首先通过 SFT 让模型学会输出符合格式要求的 Wordle 猜测，再通过 RL（强化学习）让模型学会策略性地利用游戏反馈提高猜中率。

本章是整个两阶段训练的第一站：打好 SFT 的概念基础，理解 Wordle 任务。

---

## 前置要求

为了充分掌握本章内容，你应已具备以下能力：

- 了解 PyTorch 模型训练的基本流程（forward → loss → backward → update）。
- 了解 Transformer 模型的基本结构（attention、FFN、layer norm）。
- 已访问过 Ascend NPU 环境（`npu-smi info` 可用）。

无须具备任何 Wordle 游戏或 SFT 的前置知识，本章将从零开始介绍。

---

## 章节目标

完成本章后，你将能够：

- 理解 Wordle 猜词游戏的规则和 SFT + RL 两阶段训练方案的设计动机。
- 理解 SFT 的核心概念：对话数据格式、Cross-Entropy Loss、Label Masking。
- 理解为什么 SFT 是 Wordle 训练的必要第一步。

---

## 本章内容

- [1.1 章节介绍](01.01_chapter_intro.ipynb)
- [1.2 Wordle 任务介绍](01.02_wordle_task_intro.ipynb)
- [1.3 SFT 概念与原理](01.03_sft_concepts.ipynb)
- [1.4 章节练习](01.04_chapter_practice.ipynb)

---

## 为什么需要两阶段训练？

训练模型玩 Wordle 是一个多轮决策任务：模型根据游戏反馈（哪些字母对、哪些字母位置不对）逐步缩小候选词范围，最终猜出目标单词。


```text
Wordle 游戏示例：
  目标词：APPLE
  第1轮：模型猜 CRANE → 反馈 G-Y-X-Y-X → 调整策略
  第2轮：模型猜 PLANE → 反馈 X-Y-Y-X-G → 再调整
  ...
  第N轮：模型猜 APPLE → 猜对！
```


这种多轮决策无法用单一的标准答案来训练——因为模型在第 2 轮的最佳猜测取决于第 1 轮的具体反馈，而反馈每次都可能不同。这引出了两阶段方案：

1. **SFT 阶段（本教程）**：用已有的 Wordle 游戏记录（对话数据）训练模型，让它学会：
   - 输出格式正确的猜测（`<guess>单词</guess>`）。
   - 理解 G/Y/X 反馈的含义。
   - 具备初步的猜词策略。

2. **RL 阶段（下一教程）**：让模型在实际游戏中试错，通过奖励函数优化多轮决策策略，真正提高猜中率。

> **SFT 的角色定位**：SFT 为 RL "搭好脚手架"——模型先学会输出格式正确的回复，RL 再在此基础上优化策略。没有 SFT 的格式对齐，RL 的探索空间太大，难以收敛。